# SRL v3 Research Benchmark: Span-Biaffine + Polyglot Transfer + LLM (Zero/Few-Shot & QLoRA)

**Designed for fast execution on a free Colab T4 GPU (~25–35 min total, resume-safe after every experiment).**

### Why v3 is BOTH Faster AND Architecturally Stronger than v2
1. **3x Faster Parsing & Encoding:** Deduplicates sentences across multiple predicates (PropBank-EWT drops from **40,424 → ~12,500** unique sentences!) + enables `spacy.prefer_gpu()`.
2. **True SOTA SRL Architecture (`SpanBiaffineSRLNet`):** Encodes each sentence ONCE, prunes candidate spans, scores `(predicate, span, role)` via vectorized Biaffine `einsum`, and enforces global non-overlap + unique core-role (`ARG0..ARG5`) decoding.
3. **Cross-Lingual Polyglot Transfer (RQ3):** Unifies EN (`ARG0/ARGM-*`) & ES (`A0/AM-*`) PropBank inventories into one shared schema to measure **Zero-Shot EN→ES**, **Monolingual ES**, and **Polyglot EN+ES Joint** gains.
4. **LLM Research Comparison (Encoder vs. LLM):** Evaluates **Zero-Shot**, **5-Shot (Pydantic JSON)**, and **QLoRA Fine-Tuned `Qwen2.5-3B-Instruct`** (4-bit NF4) against our 278M Span-Biaffine Encoder across **F1, JSON Schema Validity, Low-Resource Data Efficiency, and Latency (ms/inst)**.


In [ ]:
# 1. Environment setup (Run on Colab T4 / L4 / A100 GPU)
!pip install -q transformers accelerate peft bitsandbytes pydantic sentencepiece protobuf sklearn-crfsuite "spacy[transformers]" tqdm matplotlib
!python -m spacy download en_core_web_trf -q
!python -m spacy download es_dep_news_trf -q


In [ ]:
import os, sys, json, csv, time
import torch
import spacy
from google.colab import drive

drive.mount('/content/drive')
spacy.prefer_gpu()  # CRITICAL: prevents spaCy TRF models from falling back to 2-core CPU!

REPO_DIR = '/content/Semantic-Role-Labeling'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/yavuzselimuyaner/Semantic-Role-Labeling.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

PROPBANK_PATH    = '/content/drive/MyDrive/colab_upload/SRL_data/propbank_preprocessed.json'
PROPBANK_ES_PATH = '/content/drive/MyDrive/colab_upload/SRL_data/propbank_es_preprocessed.json'
OUT_DIR          = '/content/drive/MyDrive/colab_upload/SRL_data/results_v3'
os.makedirs(OUT_DIR, exist_ok=True)

RESULTS_JSON = os.path.join(OUT_DIR, 'results_v3_research.json')
if os.path.exists(RESULTS_JSON):
    with open(RESULTS_JSON, encoding='utf-8') as f:
        results = json.load(f)
    print('Resumed saved results:', list(results.keys()))
else:
    results = {}

def save_res():
    with open(RESULTS_JSON, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f'  [saved] {RESULTS_JSON}')

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
# 2. Load & unify PropBank-EWT (EN) + PropBank-ES (AnCora) into a single cross-lingual schema
from srl.data import load_dataset, truncate
from srl.schema import unify_instances, get_role_inventory, group_by_sentence, make_stratified_subset
from srl.metrics import compute_metrics, error_analysis, print_results, print_error_analysis

en_tr_raw, en_dv_raw, en_te_raw = load_dataset(PROPBANK_PATH)
es_tr_raw, es_dv_raw, es_te_raw = load_dataset(PROPBANK_ES_PATH)

en_tr = unify_instances(en_tr_raw, 'propbank')
en_dv = unify_instances(en_dv_raw, 'propbank')
en_te = unify_instances(en_te_raw, 'propbank')

es_tr = unify_instances(es_tr_raw, 'propbank_es')
es_dv = unify_instances(es_dv_raw, 'propbank_es')
es_te = unify_instances(es_te_raw, 'propbank_es')

UNIFIED_ROLES = get_role_inventory(en_tr, en_dv, en_te, es_tr, es_dv, es_te, min_count=2)
print(f'EN: {len(en_tr):,} pred instances -> {len(group_by_sentence(en_tr)):,} unique sentences')
print(f'ES: {len(es_tr):,} pred instances -> {len(group_by_sentence(es_tr)):,} unique sentences')
print(f'Unified cross-lingual role inventory (incl. NULL): {len(UNIFIED_ROLES)} roles')

# Fixed 200-instance stratified slice (100 EN + 100 ES) for fair Encoder vs LLM comparison
eval_slice_en = make_stratified_subset(en_te, n=100, seed=42)
eval_slice_es = make_stratified_subset(es_te, n=100, seed=42)
eval_slice_all = eval_slice_en + eval_slice_es


In [ ]:
# 3. Fast Bug-Fixed Baselines: Rule-Based v2 + CRF v2 (~2-3 min total with GPU spaCy + dedup!)
from srl.rule_based import RuleBasedSRL
from srl.crf_model import CRFModel, parse_aligned_docs

for ds_name, tr_raw, te_raw, spacy_name in [
    ('propbank',    en_tr_raw, en_te_raw, 'en_core_web_trf'),
    ('propbank_es', es_tr_raw, es_te_raw, 'es_dep_news_trf'),
]:
    rb_key, crf_key = f'{ds_name}_rule_v2', f'{ds_name}_crf_v2'
    if rb_key in results and crf_key in results:
        print(f'Skipping {ds_name} rule+crf (already saved)')
        continue

    nlp = spacy.load(spacy_name, disable=['ner'])
    gold = [x['labels'] for x in te_raw]
    t0 = time.time()
    test_docs = parse_aligned_docs(te_raw, nlp, batch_size=256)
    print(f'Parsed unique {ds_name} test sentences in {time.time()-t0:.1f}s')

    if rb_key not in results:
        t1 = time.time()
        rb = RuleBasedSRL(nlp=nlp, mode=ds_name)
        rb_preds = rb.predict_batch(te_raw, docs=test_docs)
        m = compute_metrics(gold, rb_preds)
        ea = error_analysis(gold, rb_preds)
        print_results(rb_key, m); print_error_analysis(rb_key, ea)
        results[rb_key] = {**m, 'error_analysis': ea, 'time_s': round(time.time()-t1, 1)}
        save_res()

    if crf_key not in results:
        t1 = time.time()
        train_docs = parse_aligned_docs(tr_raw, nlp, batch_size=256)
        print(f'Parsed unique {ds_name} train sentences in {time.time()-t1:.1f}s')
        crf = CRFModel(algorithm='l2sgd', max_iterations=80, c2=0.5, min_freq=5, all_possible_transitions=False, verbose=True)
        crf.fit(tr_raw, nlp, docs=train_docs)
        crf_preds = crf.predict(te_raw, nlp, docs=test_docs)
        m = compute_metrics(gold, crf_preds)
        ea = error_analysis(gold, crf_preds)
        print_results(crf_key, m); print_error_analysis(crf_key, ea)
        results[crf_key] = {**m, 'error_analysis': ea, 'time_s': round(time.time()-t1, 1)}
        save_res()

    del nlp
    torch.cuda.empty_cache()


In [ ]:
# 4. Multi-Predicate Span-Biaffine SRL (xlm-roberta-base, FP16) — Monolingual, Zero-Shot EN->ES, & Polyglot EN+ES
# (~15 min total on Colab T4 because each unique sentence is encoded ONCE!)
from srl.span_biaffine import SpanBiaffineTrainer

gold_en_te = [x['labels'] for x in en_te]
gold_es_te = [x['labels'] for x in es_te]
gold_slice_en = [x['labels'] for x in eval_slice_en]
gold_slice_es = [x['labels'] for x in eval_slice_es]

# --- Exp A: EN Monolingual + Zero-Shot EN -> ES ---
if 'span_biaffine_en_mono' not in results:
    t0 = time.time()
    tr_en = SpanBiaffineTrainer(UNIFIED_ROLES, model_name='xlm-roberta-base', batch_size=16, epochs=6, backbone_lr=2.5e-5, head_lr=1e-3)
    hist = tr_en.fit(en_tr, dev_instances=en_dv, save_path=os.path.join(OUT_DIR, 'span_xlmr_en.pt'))
    en_preds, en_ms = tr_en.predict_with_latency(en_te)
    zs_es_preds, zs_ms = tr_en.predict_with_latency(es_te)
    m_en = compute_metrics(gold_en_te, en_preds)
    m_zs = compute_metrics(gold_es_te, zs_es_preds)
    print_results('Span-Biaffine XLM-R [EN Mono -> EN]', m_en)
    print_results('Span-Biaffine XLM-R [Zero-Shot EN -> ES]', m_zs)
    results['span_biaffine_en_mono'] = {**m_en, 'latency_ms': en_ms, 'history': hist, 'time_s': round(time.time()-t0, 1)}
    results['span_biaffine_zeroshot_en_to_es'] = {**m_zs, 'latency_ms': zs_ms}
    save_res()
    del tr_en; torch.cuda.empty_cache()

# --- Exp B: ES Monolingual ---
if 'span_biaffine_es_mono' not in results:
    t0 = time.time()
    tr_es = SpanBiaffineTrainer(UNIFIED_ROLES, model_name='xlm-roberta-base', batch_size=16, epochs=6, backbone_lr=2.5e-5, head_lr=1e-3)
    hist = tr_es.fit(es_tr, dev_instances=es_dv, save_path=os.path.join(OUT_DIR, 'span_xlmr_es.pt'))
    es_preds, es_ms = tr_es.predict_with_latency(es_te)
    m_es = compute_metrics(gold_es_te, es_preds)
    print_results('Span-Biaffine XLM-R [ES Mono -> ES]', m_es)
    results['span_biaffine_es_mono'] = {**m_es, 'latency_ms': es_ms, 'history': hist, 'time_s': round(time.time()-t0, 1)}
    save_res()
    del tr_es; torch.cuda.empty_cache()

# --- Exp C: Polyglot EN + ES Joint Training ---
if 'span_biaffine_polyglot_en' not in results:
    t0 = time.time()
    tr_poly = SpanBiaffineTrainer(UNIFIED_ROLES, model_name='xlm-roberta-base', batch_size=16, epochs=6, backbone_lr=2.5e-5, head_lr=1e-3)
    hist = tr_poly.fit(en_tr + es_tr, dev_instances=en_dv + es_dv, save_path=os.path.join(OUT_DIR, 'span_xlmr_polyglot.pt'))
    p_en, ms_en = tr_poly.predict_with_latency(en_te)
    p_es, ms_es = tr_poly.predict_with_latency(es_te)
    # Also evaluate on exact 200-sent LLM benchmark slice for apples-to-apples comparison
    s_en, ms_s_en = tr_poly.predict_with_latency(eval_slice_en, show_progress=False)
    s_es, ms_s_es = tr_poly.predict_with_latency(eval_slice_es, show_progress=False)
    m_en = compute_metrics(gold_en_te, p_en)
    m_es = compute_metrics(gold_es_te, p_es)
    print_results('Span-Biaffine XLM-R [Polyglot -> EN]', m_en)
    print_results('Span-Biaffine XLM-R [Polyglot -> ES]', m_es)
    results['span_biaffine_polyglot_en'] = {**m_en, 'latency_ms': ms_en, 'error_analysis': error_analysis(gold_en_te, p_en), 'history': hist, 'time_s': round(time.time()-t0, 1)}
    results['span_biaffine_polyglot_es'] = {**m_es, 'latency_ms': ms_es, 'error_analysis': error_analysis(gold_es_te, p_es)}
    results['slice_span_biaffine_polyglot_en'] = {**compute_metrics(gold_slice_en, s_en), 'latency_ms': ms_s_en}
    results['slice_span_biaffine_polyglot_es'] = {**compute_metrics(gold_slice_es, s_es), 'latency_ms': ms_s_es}
    save_res()
    del tr_poly; torch.cuda.empty_cache()


In [ ]:
# 5. Low-Resource Data-Efficiency Curve for Span-Biaffine Encoder (500 examples, ~1 min on T4)
# Matches the EXACT 500-example budget we give QLoRA Qwen2.5-3B in the next cell!
low_res_train = truncate(en_tr, 250, seed=42) + truncate(es_tr, 250, seed=42)

if 'slice_span_biaffine_500shot_en' not in results:
    t0 = time.time()
    tr_low = SpanBiaffineTrainer(UNIFIED_ROLES, model_name='xlm-roberta-base', batch_size=16, epochs=12, backbone_lr=3e-5, head_lr=1e-3, patience=4)
    tr_low.fit(low_res_train, dev_instances=en_dv[:500] + es_dv[:500])
    s_en, ms_en = tr_low.predict_with_latency(eval_slice_en, show_progress=False)
    s_es, ms_es = tr_low.predict_with_latency(eval_slice_es, show_progress=False)
    m_en = compute_metrics(gold_slice_en, s_en)
    m_es = compute_metrics(gold_slice_es, s_es)
    print_results('Span-Biaffine 500-shot [EN slice]', m_en)
    print_results('Span-Biaffine 500-shot [ES slice]', m_es)
    results['slice_span_biaffine_500shot_en'] = {**m_en, 'latency_ms': ms_en, 'train_s': round(time.time()-t0, 1)}
    results['slice_span_biaffine_500shot_es'] = {**m_es, 'latency_ms': ms_es}
    save_res()
    del tr_low; torch.cuda.empty_cache()


In [ ]:
# 6. LLM Research Benchmark: Zero-Shot, 5-Shot (Pydantic JSON), and 500-Shot QLoRA on Qwen2.5-3B-Instruct
# (~10-12 min total on Colab T4 in 4-bit NF4!)
from srl.llm_srl import LocalHFLlmSRL

LLM_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
few_shot_en = make_stratified_subset(en_tr, n=5, seed=7)
few_shot_es = make_stratified_subset(es_tr, n=5, seed=7)

need_llm = any(
    k not in results
    for k in ['slice_llm_0shot_en', 'slice_llm_5shot_en', 'slice_llm_qlora500_en']
)

if need_llm:
    llm = LocalHFLlmSRL(model_name=LLM_MODEL, load_in_4bit=True, max_new_tokens=220)

    if 'slice_llm_0shot_en' not in results:
        results['slice_llm_0shot_en'] = llm.evaluate(eval_slice_en, few_shot_examples=None, batch_size=8)
        results['slice_llm_0shot_es'] = llm.evaluate(eval_slice_es, few_shot_examples=None, batch_size=8)
        print('0-shot EN:', results['slice_llm_0shot_en']['f1'], 'JSON valid:', results['slice_llm_0shot_en']['json_valid_rate'])
        print('0-shot ES:', results['slice_llm_0shot_es']['f1'], 'JSON valid:', results['slice_llm_0shot_es']['json_valid_rate'])
        save_res()

    if 'slice_llm_5shot_en' not in results:
        results['slice_llm_5shot_en'] = llm.evaluate(eval_slice_en, few_shot_examples=few_shot_en, batch_size=4)
        results['slice_llm_5shot_es'] = llm.evaluate(eval_slice_es, few_shot_examples=few_shot_es, batch_size=4)
        print('5-shot EN:', results['slice_llm_5shot_en']['f1'], 'JSON valid:', results['slice_llm_5shot_en']['json_valid_rate'])
        print('5-shot ES:', results['slice_llm_5shot_es']['f1'], 'JSON valid:', results['slice_llm_5shot_es']['json_valid_rate'])
        save_res()

    if 'slice_llm_qlora500_en' not in results:
        train_s = llm.finetune_qlora(low_res_train, epochs=2, batch_size=4, grad_accum=4, lr=2e-4, lora_r=16)
        results['slice_llm_qlora500_en'] = {**llm.evaluate(eval_slice_en, few_shot_examples=None, batch_size=8), 'train_s': train_s}
        results['slice_llm_qlora500_es'] = {**llm.evaluate(eval_slice_es, few_shot_examples=None, batch_size=8), 'train_s': train_s}
        print('QLoRA 500-shot EN:', results['slice_llm_qlora500_en']['f1'], 'JSON valid:', results['slice_llm_qlora500_en']['json_valid_rate'])
        print('QLoRA 500-shot ES:', results['slice_llm_qlora500_es']['f1'], 'JSON valid:', results['slice_llm_qlora500_es']['json_valid_rate'])
        save_res()

    del llm; torch.cuda.empty_cache()


In [ ]:
# 7. Print Master Summary Table (Full Test + 200-Sentence Encoder vs LLM Slice)
print('\n' + '=' * 88)
print(f'  {"Experiment":<36} {"Prec":>7} {"Rec":>7} {"F1":>7} {"JSON%":>7} {"ms/inst":>9}')
print('-' * 88)
rows = []
for name, r in results.items():
    jv = f"{r['json_valid_rate']*100:.1f}%" if 'json_valid_rate' in r else '  N/A'
    lat = r.get('latency_ms', r.get('avg_latency_ms', None))
    lat_s = f'{lat:.1f}' if lat is not None else 'N/A'
    print(f'  {name:<36} {r["precision"]:>7.4f} {r["recall"]:>7.4f} {r["f1"]:>7.4f} {jv:>7} {lat_s:>9}')
    rows.append({
        'experiment': name, 'precision': r['precision'], 'recall': r['recall'], 'f1': r['f1'],
        'json_valid_rate': r.get('json_valid_rate', ''), 'latency_ms': lat if lat is not None else '',
    })
print('=' * 88)

csv_path = os.path.join(OUT_DIR, 'results_v3_summary.csv')
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader(); w.writerows(rows)
print('Exported:', csv_path)
